
# SQL 基础：建库、查询、聚合

用 SQLite 建一个 5 条记录的小表，把最常用的四种查询跑一遍。
SQLite 是 Python 自带的，不需要装任何数据库服务。

这个 notebook 同时是下一个项目（Text-to-SQL）的数据准备步骤——它会在当前目录生成 `company.db`。


In [1]:

import sqlite3

import pandas as pd

# 连接数据库，文件不存在时自动创建
conn = sqlite3.connect("company.db")
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS employees (
        id         INTEGER PRIMARY KEY,
        name       TEXT,
        department TEXT,
        salary     INTEGER
    )
''')

# 先清空，保证多次运行结果一致
cursor.execute("DELETE FROM employees")
cursor.executemany(
    "INSERT INTO employees VALUES (?, ?, ?, ?)",
    [
        (1, "张三", "研发部", 15000),
        (2, "李四", "研发部", 18000),
        (3, "王五", "销售部", 12000),
        (4, "赵六", "销售部", 20000),
        (5, "孙七", "人事部", 9000),
    ],
)
conn.commit()
print("数据库准备完成：company.db")


数据库准备完成：company.db



## 查询 1：全表

`SELECT *` 查所有列。


In [2]:

print(pd.read_sql_query("SELECT * FROM employees", conn))


   id name department  salary
0   1   张三        研发部   15000
1   2   李四        研发部   18000
2   3   王五        销售部   12000
3   4   赵六        销售部   20000
4   5   孙七        人事部    9000



## 查询 2：条件筛选

`WHERE` 按条件过滤行。


In [3]:

print(pd.read_sql_query("SELECT * FROM employees WHERE department = '研发部'", conn))


   id name department  salary
0   1   张三        研发部   15000
1   2   李四        研发部   18000


In [4]:

# 多条条件用 AND 连起来
query = "SELECT name, salary FROM employees WHERE salary > 15000"
print(pd.read_sql_query(query, conn))


  name  salary
0   李四   18000
1   赵六   20000



## 查询 3：聚合

`GROUP BY` 分组，`AVG()` / `COUNT()` 等聚合函数对每组做统计。


In [5]:

query = '''
    SELECT department, AVG(salary) AS avg_salary, COUNT(*) AS headcount
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
'''
print(pd.read_sql_query(query, conn))


  department  avg_salary  headcount
0        研发部     16500.0          2
1        销售部     16000.0          2
2        人事部      9000.0          1



## 小结

| 需求 | 关键字 |
| --- | --- |
| 查哪些列 | `SELECT` |
| 查哪些行 | `WHERE` |
| 怎么分组统计 | `GROUP BY` + 聚合函数 |
| 结果排序 | `ORDER BY` |

`SELECT * FROM employees` 里的 `*` 表示"所有列"。注意 `SELECT` 和 `*` 之间要有空格——
`SELECT*FROM` 虽然多数数据库也能解析，但可读性差，写规范一点。


In [6]:

conn.close()
print("连接已关闭")


连接已关闭
